# Exploratory Data Analysis and Data-Driven Decision-Making for Agribusiness
**Internship Project** | Dataset: `agri_data.csv` | Notebook v1.0

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — PROJECT INTRODUCTION
# ══════════════════════════════════════════════════════════════════════════════

print('''
OBJECTIVE
=========
Analyse the agri_data.csv dataset to extract actionable insights on:
  - Crop yield patterns across crops, regions, and time
  - Cultivated area distribution and production estimates
  - Crop prices and their variability
  - Weather conditions and their relationship with yield
  - Regional performance differences
  - Data quality issues that could affect decisions

BUSINESS QUESTIONS
==================
  1. Which crops have the highest and lowest average yields?
  2. Which regions show the highest and lowest average yields?
  3. How much does yield vary across crops or regions?
  4. Which crops command higher or lower recorded prices?
  5. What relationships appear between weather conditions and yield?
  6. What production patterns are visible in the data?
  7. What data quality issues might affect decision-making?
  8. What additional data would improve future analysis?

WORKFLOW
========
  Load -> Inspect -> Quality Assessment -> Clean -> Feature Engineering
  -> Descriptive Stats -> Univariate -> Bivariate -> Time-Series
  -> Correlation -> Outliers -> Business Questions -> Recommendations
  -> KPIs -> Summary -> Export
''')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — IMPORT LIBRARIES
# ══════════════════════════════════════════════════════════════════════════════
# If any library is missing, run:  !pip install pandas numpy matplotlib seaborn

import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import matplotlib

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

plt.rcParams.update({
    'figure.dpi': 120, 'figure.facecolor': '#F8F9FA',
    'axes.facecolor': '#FFFFFF', 'axes.grid': True, 'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans', 'axes.titlesize': 13,
    'axes.labelsize': 11, 'xtick.labelsize': 9, 'ytick.labelsize': 9,
})
sns.set_palette('Set2')

print(f'pandas     {pd.__version__}')
print(f'numpy      {np.__version__}')
print(f'matplotlib {matplotlib.__version__}')
print(f'seaborn    {sns.__version__}')
print('All libraries loaded.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — LOAD THE DATASET
# ══════════════════════════════════════════════════════════════════════════════

DATA_FILE  = Path('agri_data.csv')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"'{DATA_FILE}' not found. "
        "Place agri_data.csv next to this notebook and re-run."
    )

raw_df = pd.read_csv(DATA_FILE)
print(f'Loaded: {DATA_FILE.resolve()}')
print(f'Shape : {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print()

# Helper to save figures
def save_fig(name):
    p = OUTPUT_DIR / name
    plt.savefig(p, bbox_inches='tight', dpi=150)
    print(f'  Saved -> {p}')

In [ ]:
# First 5 rows
raw_df.head()

In [ ]:
# Last 5 rows
raw_df.tail()

In [ ]:
# Column names and types
print('Column names and data types:')
for i, (col, dtype) in enumerate(raw_df.dtypes.items(), 1):
    print(f'  {i:>2}. {col:<26} {dtype}')

print(f'''
DATASET STRUCTURE
=================
The dataset has {raw_df.shape[0]:,} field-level agricultural observations across
{raw_df.shape[1]} columns, spanning years {raw_df["season"].min()} to {raw_df["season"].max()}.

Crops   : {sorted(raw_df["crop"].unique())}
Regions : {sorted(raw_df["region"].unique())}

Extra columns beyond specification (used to enrich analysis):
  flowering_rainfall_mm, heat_stress_days, irrigated, variety,
  soil_type, fertiliser_kg_ha, latitude, longitude
''')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — INITIAL DATA INSPECTION
# ══════════════════════════════════════════════════════════════════════════════

print(f'Shape: {raw_df.shape}\n')

# Unique values per column
unique_df = pd.DataFrame({
    'Data Type': raw_df.dtypes,
    'Unique': raw_df.nunique(),
    'Missing': raw_df.isnull().sum(),
    'Missing %': (raw_df.isnull().sum() / len(raw_df) * 100).round(2)
})
print('Column overview:')
print(unique_df.to_string())

print(f'\nExact duplicate rows: {raw_df.duplicated().sum()}')

In [ ]:
# Numerical summary
print('Numerical summary statistics:')
raw_df.describe(include=[np.number]).T

In [ ]:
# Categorical summary
print('Categorical summary statistics:')
raw_df.describe(include='object').T

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — DATA QUALITY ASSESSMENT
# ══════════════════════════════════════════════════════════════════════════════

quality_issues = []

def log_issue(issue, column, n, treatment):
    quality_issues.append({
        'Issue': issue, 'Column': column,
        'Affected Records': n, 'Treatment / Action': treatment
    })

# 5.1 Missing values
for col in raw_df.columns:
    n = raw_df[col].isnull().sum()
    if n > 0:
        log_issue('Missing value', col, n,
                  'Impute: median (numeric) or mode (categorical)')

# 5.2 Duplicates
n_dupes = raw_df.duplicated().sum()
if n_dupes > 0:
    log_issue('Exact duplicate row', 'All columns', n_dupes, 'Remove')

# 5.3 Data-type checks
season_bad = pd.to_numeric(raw_df['season'], errors='coerce').isna().sum()
if season_bad > 0:
    log_issue('Non-numeric season', 'season', season_bad, 'Convert to int; invalid -> NaN')

irr_bad = (~raw_df['irrigated'].isin([0, 1])).sum()
if irr_bad > 0:
    log_issue('Invalid irrigated flag', 'irrigated', irr_bad, 'Investigate')

# 5.4 Whitespace
for col in raw_df.select_dtypes(include='object').columns:
    n_ws = (raw_df[col].dropna().str.strip() != raw_df[col].dropna()).sum()
    if n_ws > 0:
        log_issue('Whitespace', col, n_ws, 'Strip')

# 5.5 Plausibility (extreme values flagged, NOT auto-removed)
bounds = {
    'yield_t_ha':       (0,  20),  'area_ha':          (0,  500),
    'price_usd_per_t':  (0,  3000),'rainfall_mm':      (0,  2000),
    'tmax_c':           (15, 45),  'tmin_c':           (-5, 35),
    'fertiliser_kg_ha': (0,  600), 'heat_stress_days': (0,  180),
}
for col, (lo, hi) in bounds.items():
    n = ((raw_df[col] < lo) | (raw_df[col] > hi)).sum()
    if n > 0:
        log_issue(f'{col} outside [{lo},{hi}]', col, n,
                  'Flag for investigation; do not auto-remove')

# 5.6 Category name inspection
print('Category values:')
for col in ['crop', 'region', 'soil_type', 'variety']:
    vals = sorted(raw_df[col].dropna().unique())
    print(f'  {col} ({len(vals)}): {vals}')

# 5.7 Quality summary table
quality_df = pd.DataFrame(quality_issues)
print(f'\nData Quality Summary — {len(quality_df)} issue(s):')
if not quality_df.empty:
    print(quality_df.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — DATA CLEANING
# ══════════════════════════════════════════════════════════════════════════════
# Rule: raw_df is NEVER modified. All cleaning applied to working copy `df`.

df = raw_df.copy()
print(f'Starting rows: {len(df):,}')

# 6.1 Strip whitespace from text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()
print('  -> Whitespace stripped from all text columns.')

# 6.2 Convert season to integer year
df['year'] = pd.to_numeric(df['season'], errors='coerce').astype('Int64')
print(f'  -> year column created. Range: {df["year"].min()}-{df["year"].max()}. '
      f'Invalid: {df["year"].isna().sum()}')

# 6.3 Remove exact duplicate rows
n_before = len(df)
df = df.drop_duplicates()
n_removed = n_before - len(df)
print(f'  -> Duplicates removed: {n_removed}. Rows remaining: {len(df):,}')

# 6.4 Handle missing values
# Only 24 rows (0.2%) missing soil_type and fertiliser_kg_ha.
# Strategy: mode for categorical, median for numeric.
# DOMAIN FLAG: A GIS-based soil lookup would be more accurate.
n_soil = df['soil_type'].isna().sum()
n_fert = df['fertiliser_kg_ha'].isna().sum()
soil_mode   = df['soil_type'].mode()[0]
fert_median = df['fertiliser_kg_ha'].median()
df['soil_type']        = df['soil_type'].fillna(soil_mode)
df['fertiliser_kg_ha'] = df['fertiliser_kg_ha'].fillna(fert_median)
print(f'  -> soil_type: {n_soil} missing filled with mode "{soil_mode}"')
print(f'  -> fertiliser_kg_ha: {n_fert} missing filled with median {fert_median:.1f} kg/ha')

# 6.5 Create readable irrigation label
df['irrigated_label'] = df['irrigated'].map({0: 'Non-Irrigated', 1: 'Irrigated'})
print(f'  -> irrigated_label created: {df["irrigated_label"].value_counts().to_dict()}')

# 6.6 No outliers removed — investigated in Section 13
print(f'  -> No outliers removed (investigated in Section 13).')
print(f'\nFinal cleaned rows: {len(df):,}')
print(f'Remaining missing: {df.isnull().sum().sum()}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════

# 7.1 Production (tonnes) = yield_t_ha * area_ha
# Units: (t/ha) * (ha) = tonnes — compatible.
df['production_t'] = df['yield_t_ha'] * df['area_ha']
print(f'production_t: {df["production_t"].min():.1f} - {df["production_t"].max():.1f} tonnes')

# 7.2 Temperature range = tmax - tmin
df['temp_range_c'] = df['tmax_c'] - df['tmin_c']
print(f'temp_range_c: {df["temp_range_c"].min():.1f} - {df["temp_range_c"].max():.1f} C')

# 7.3 Gross crop value ESTIMATE = production * price
# WARNING: This is NOT verified revenue — no cost data available.
df['gross_value_usd'] = df['production_t'] * df['price_usd_per_t']
print(f'gross_value_usd (ESTIMATE): ${df["gross_value_usd"].min():,.0f} - ${df["gross_value_usd"].max():,.0f}')

# 7.4 Heat stress category
df['heat_stress_cat'] = pd.cut(
    df['heat_stress_days'],
    bins=[-1, 5, 15, df['heat_stress_days'].max() + 1],
    labels=['Low (0-5)', 'Medium (6-15)', 'High (>15)']
)
print(f'heat_stress_cat: {df["heat_stress_cat"].value_counts().sort_index().to_dict()}')

# 7.5 Verify engineered features
assert (df['production_t'] >= 0).all(), 'Negative production detected!'
assert (df['temp_range_c'] >= 0).all(), 'Negative temp range detected!'
print('\nAll feature validations passed.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — DESCRIPTIVE STATISTICAL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def describe_col(series):
    """Compact descriptive stats for a single numeric Series."""
    return {
        'Mean': series.mean(), 'Median': series.median(),
        'Min': series.min(), 'Max': series.max(),
        'Std': series.std(), 'P25': series.quantile(0.25),
        'P75': series.quantile(0.75), 'P90': series.quantile(0.90),
    }

num_cols = ['yield_t_ha', 'area_ha', 'price_usd_per_t', 'rainfall_mm',
            'tmax_c', 'tmin_c', 'heat_stress_days', 'fertiliser_kg_ha', 'production_t']

print('8.1 Overall descriptive statistics:')
overall_stats = pd.DataFrame([describe_col(df[c]) for c in num_cols], index=num_cols).round(3)
print(overall_stats.to_string())

In [ ]:
# 8.2 Average yield by crop
yield_by_crop = (
    df.groupby('crop')['yield_t_ha']
      .agg(Mean='mean', Median='median', Std='std', Min='min', Max='max', Count='count')
      .round(3).sort_values('Mean', ascending=False)
)
print('Average yield by crop (t/ha):')
print(yield_by_crop.to_string())

In [ ]:
# 8.3 Average yield by region
yield_by_region = (
    df.groupby('region')['yield_t_ha']
      .agg(Mean='mean', Median='median', Std='std', Min='min', Max='max', Count='count')
      .round(3).sort_values('Mean', ascending=False)
)
print('Average yield by region (t/ha):')
print(yield_by_region.to_string())

In [ ]:
# 8.4 Average price by crop
price_by_crop = (
    df.groupby('crop')['price_usd_per_t']
      .agg(Mean='mean', Median='median', Std='std', Min='min', Max='max')
      .round(2).sort_values('Mean', ascending=False)
)
print('Average price by crop (USD/t):')
print(price_by_crop.to_string())

In [ ]:
# 8.5 Estimated production by crop
prod_by_crop = (
    df.groupby('crop')['production_t']
      .agg(Total='sum', Mean='mean', Median='median')
      .round(1).sort_values('Total', ascending=False)
)
print('Estimated production by crop (tonnes):')
print(prod_by_crop.to_string())

In [ ]:
# 8.6 Yield variability — Coefficient of Variation
# CV = Std / Mean * 100  —  higher = more production risk
cv_by_crop = df.groupby('crop')['yield_t_ha'].agg(Mean='mean', Std='std')
cv_by_crop['CV_%'] = (cv_by_crop['Std'] / cv_by_crop['Mean'] * 100).round(1)
cv_by_crop = cv_by_crop.sort_values('CV_%', ascending=False)
print('Yield variability (CV) by crop:')
print(cv_by_crop.to_string())
print(f'\nHighest CV: {cv_by_crop["CV_%"].idxmax()} ({cv_by_crop["CV_%"].max():.1f}%)')
print(f'Lowest CV : {cv_by_crop["CV_%"].idxmin()} ({cv_by_crop["CV_%"].min():.1f}%)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 9 — UNIVARIATE ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

# 9.1 Key numeric distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Univariate Distributions — Key Numeric Variables', fontsize=14, fontweight='bold')

for ax, (col, title, xlabel) in zip(axes.flat, [
    ('yield_t_ha',      'Crop Yield',       'Yield (t/ha)'),
    ('price_usd_per_t', 'Crop Price',        'Price (USD/t)'),
    ('area_ha',         'Cultivated Area',   'Area (ha)'),
    ('rainfall_mm',     'Seasonal Rainfall', 'Rainfall (mm)'),
]):
    sns.histplot(df[col].dropna(), kde=True, bins=40, ax=ax, color='steelblue')
    ax.axvline(df[col].mean(),   color='red',    ls='--', lw=1.2, label=f'Mean {df[col].mean():.2f}')
    ax.axvline(df[col].median(), color='orange', ls=':',  lw=1.2, label=f'Median {df[col].median():.2f}')
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)

plt.tight_layout(); save_fig('09a_univariate_distributions.png'); plt.show()

In [ ]:
# 9.2 Temperature and heat-stress distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Temperature and Heat-Stress Distributions', fontsize=13, fontweight='bold')

for ax, (col, label) in zip(axes, [
    ('tmax_c', 'Max Temp (C)'), ('tmin_c', 'Min Temp (C)'), ('heat_stress_days', 'Heat-Stress Days'),
]):
    sns.histplot(df[col].dropna(), kde=True, bins=35, ax=ax, color='tomato')
    ax.set_xlabel(label); ax.set_ylabel('Frequency'); ax.set_title(label)

plt.tight_layout(); save_fig('09b_temperature_distributions.png'); plt.show()

In [ ]:
# 9.3 Crop and region frequency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Record Counts by Crop and Region', fontsize=13, fontweight='bold')

crop_counts   = df['crop'].value_counts()
region_counts = df['region'].value_counts()

bars1 = axes[0].bar(crop_counts.index, crop_counts.values,
                    color=sns.color_palette('Set2', len(crop_counts)))
axes[0].set_title('Records per Crop'); axes[0].set_xlabel('Crop'); axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for b in bars1:
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+10,
                 f'{int(b.get_height()):,}', ha='center', va='bottom', fontsize=8)

bars2 = axes[1].barh(region_counts.index, region_counts.values,
                     color=sns.color_palette('Paired', len(region_counts)))
axes[1].set_title('Records per Region'); axes[1].set_xlabel('Count')
for b in bars2:
    axes[1].text(b.get_width()+5, b.get_y()+b.get_height()/2,
                 f'{int(b.get_width()):,}', va='center', fontsize=8)

plt.tight_layout(); save_fig('09c_crop_region_counts.png'); plt.show()

In [ ]:
# 9.4 Irrigation and fertiliser
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Irrigation Status and Fertiliser Application', fontsize=13, fontweight='bold')

irr_counts = df['irrigated_label'].value_counts()
axes[0].pie(irr_counts, labels=irr_counts.index, autopct='%1.1f%%',
            colors=['#2196F3', '#FF9800'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Irrigation Status')

sns.histplot(df['fertiliser_kg_ha'].dropna(), bins=40, kde=True, ax=axes[1], color='#4CAF50')
axes[1].set_title('Fertiliser Application'); axes[1].set_xlabel('Fertiliser (kg/ha)')

plt.tight_layout(); save_fig('09d_irrigation_fertiliser.png'); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 10 — BIVARIATE ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

crop_order   = df.groupby('crop')['yield_t_ha'].median().sort_values(ascending=False).index
region_order = df.groupby('region')['yield_t_ha'].median().sort_values(ascending=False).index

# 10.1 Yield by crop
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df, x='crop', y='yield_t_ha', order=crop_order, palette='Set2', ax=ax)
ax.set_title('Yield Distribution by Crop (sorted by median)', fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Yield (t/ha)')
plt.tight_layout(); save_fig('10a_yield_by_crop_boxplot.png'); plt.show()

In [ ]:
# 10.2 Yield by region
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=df, x='region', y='yield_t_ha', order=region_order, palette='Paired', ax=ax)
ax.set_title('Yield Distribution by Region (sorted by median)', fontweight='bold')
ax.set_xlabel('Region'); ax.set_ylabel('Yield (t/ha)'); ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); save_fig('10b_yield_by_region_boxplot.png'); plt.show()

In [ ]:
# 10.3 Average price by crop
price_order = df.groupby('crop')['price_usd_per_t'].mean().sort_values(ascending=False).index
avg_price   = df.groupby('crop')['price_usd_per_t'].mean().loc[price_order]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(avg_price.index, avg_price.values, color=sns.color_palette('Set3', len(avg_price)))
for b, v in zip(bars, avg_price.values):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+5, f'${v:,.0f}', ha='center', fontsize=9)
ax.set_title('Average Crop Price (USD/tonne)', fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Average Price (USD/t)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout(); save_fig('10c_avg_price_by_crop.png'); plt.show()

In [ ]:
# 10.4 Rainfall vs Yield
fig, ax = plt.subplots(figsize=(10, 6))
for name, grp in df.groupby('crop'):
    ax.scatter(grp['rainfall_mm'], grp['yield_t_ha'], alpha=0.35, s=15, label=name)
ax.set_title('Rainfall vs Yield by Crop', fontweight='bold')
ax.set_xlabel('Seasonal Rainfall (mm)'); ax.set_ylabel('Yield (t/ha)')
ax.legend(title='Crop', fontsize=8, markerscale=2)
plt.tight_layout(); save_fig('10d_rainfall_vs_yield.png'); plt.show()

In [ ]:
# 10.5 Temperature vs Yield
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Temperature vs Yield', fontsize=13, fontweight='bold')

for ax, (col, xlabel) in zip(axes, [('tmax_c', 'Max Temp (C)'), ('tmin_c', 'Min Temp (C)')]):
    ax.scatter(df[col], df['yield_t_ha'], alpha=0.3, s=10, color='coral')
    ax.set_xlabel(xlabel); ax.set_ylabel('Yield (t/ha)'); ax.set_title(f'{xlabel} vs Yield')

plt.tight_layout(); save_fig('10e_temperature_vs_yield.png'); plt.show()

In [ ]:
# 10.6 Area vs Production
fig, ax = plt.subplots(figsize=(10, 6))
for name, grp in df.groupby('crop'):
    ax.scatter(grp['area_ha'], grp['production_t'], alpha=0.3, s=14, label=name)
ax.set_title('Cultivated Area vs Estimated Production', fontweight='bold')
ax.set_xlabel('Area (ha)'); ax.set_ylabel('Estimated Production (tonnes)')
ax.legend(title='Crop', fontsize=8, markerscale=2)
plt.tight_layout(); save_fig('10f_area_vs_production.png'); plt.show()

In [ ]:
# 10.7 Irrigated vs Non-Irrigated yield
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=df, x='crop', y='yield_t_ha', hue='irrigated_label',
            order=crop_order, palette=['#FF9800', '#2196F3'], ax=ax)
ax.set_title('Yield by Crop: Irrigated vs Non-Irrigated', fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Yield (t/ha)'); ax.legend(title='Irrigation')
plt.tight_layout(); save_fig('10g_yield_irrigated_vs_not.png'); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 11 — TIME-SERIES ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
# Season contains 4-digit year strings (2005-2024) — valid for time-series.

yearly = (
    df.groupby('year').agg(
        mean_yield    = ('yield_t_ha',      'mean'),
        mean_price    = ('price_usd_per_t', 'mean'),
        mean_rainfall = ('rainfall_mm',     'mean'),
        mean_tmax     = ('tmax_c',          'mean'),
        total_prod    = ('production_t',    'sum'),
        n_records     = ('yield_t_ha',      'count'),
    ).reset_index()
)
print(f'Years: {int(yearly["year"].min())} to {int(yearly["year"].max())} ({len(yearly)} years)')
print(yearly.to_string(index=False))

In [ ]:
# 11.2 Yield trend
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly['year'], yearly['mean_yield'], marker='o', lw=2, color='steelblue', label='Mean Yield')
ax.fill_between(yearly['year'],
                yearly['mean_yield'] - yearly['mean_yield'].std(),
                yearly['mean_yield'] + yearly['mean_yield'].std(),
                alpha=0.15, color='steelblue', label='+/- 1 SD')
ax.set_title('Average Crop Yield Over Time (2005-2024)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Yield (t/ha)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2)); ax.legend()
plt.tight_layout(); save_fig('11a_yield_trend.png'); plt.show()

In [ ]:
# 11.3 Price trend
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly['year'], yearly['mean_price'], marker='s', lw=2, color='darkorange')
ax.set_title('Average Crop Price Over Time (2005-2024)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Price (USD/t)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.xaxis.set_major_locator(mticker.MultipleLocator(2))
plt.tight_layout(); save_fig('11b_price_trend.png'); plt.show()

In [ ]:
# 11.4 Per-crop yield trend
yearly_crop = df.groupby(['year', 'crop'])['yield_t_ha'].mean().reset_index()

fig, ax = plt.subplots(figsize=(13, 6))
for name, grp in yearly_crop.groupby('crop'):
    ax.plot(grp['year'], grp['yield_t_ha'], marker='o', lw=1.8, markersize=4, label=name)
ax.set_title('Yield Trend by Crop (2005-2024)', fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Mean Yield (t/ha)')
ax.xaxis.set_major_locator(mticker.MultipleLocator(2)); ax.legend(title='Crop', fontsize=8)
plt.tight_layout(); save_fig('11c_yield_trend_by_crop.png'); plt.show()

In [ ]:
# 11.5 Weather trends
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Rainfall and Temperature Trends (2005-2024)', fontsize=13, fontweight='bold')

axes[0].plot(yearly['year'], yearly['mean_rainfall'], marker='o', color='teal', lw=2)
axes[0].set_title('Mean Seasonal Rainfall'); axes[0].set_xlabel('Year'); axes[0].set_ylabel('Rainfall (mm)')
axes[0].xaxis.set_major_locator(mticker.MultipleLocator(2))

axes[1].plot(yearly['year'], yearly['mean_tmax'], marker='o', color='firebrick', lw=2)
axes[1].set_title('Mean Max Temperature'); axes[1].set_xlabel('Year'); axes[1].set_ylabel('Tmax (C)')
axes[1].xaxis.set_major_locator(mticker.MultipleLocator(2))

plt.tight_layout(); save_fig('11d_weather_trends.png'); plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 12 — CORRELATION ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════
# NOTE: Correlation != causation. A weather-yield correlation does NOT prove
# that weather causes yield changes.

corr_cols = ['yield_t_ha', 'area_ha', 'price_usd_per_t', 'rainfall_mm',
             'flowering_rainfall_mm', 'tmax_c', 'tmin_c', 'temp_range_c',
             'heat_stress_days', 'fertiliser_kg_ha', 'production_t']

corr_data = df[corr_cols].dropna()
corr_mat  = corr_data.corr()
print(f'Correlation computed on {len(corr_data):,} complete observations '
      f'({len(df) - len(corr_data)} excluded for missing data).')

In [ ]:
# 12.2 Heatmap
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Correlation Matrix — Pearson r (lower triangle)', fontweight='bold')
plt.tight_layout(); save_fig('12a_correlation_heatmap.png'); plt.show()

In [ ]:
# 12.3 Top correlations with yield
yield_corr = (corr_mat['yield_t_ha'].drop('yield_t_ha')
              .sort_values(key=abs, ascending=False))
print('Top correlations with yield_t_ha:')
for var, r in yield_corr.items():
    print(f'  {var:<26} r = {r:+.3f}')

print('\nInterpretation: production_t has near-perfect correlation with yield because it is')
print('derived from it. For independent variables, check within-crop correlations for')
print('stronger patterns. Correlation does NOT prove causation.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 13 — OUTLIER INVESTIGATION
# ══════════════════════════════════════════════════════════════════════════════

outlier_cols = ['yield_t_ha', 'area_ha', 'price_usd_per_t',
                'rainfall_mm', 'fertiliser_kg_ha', 'heat_stress_days']

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fig.suptitle('Outlier Investigation (Box Plots)', fontsize=13, fontweight='bold')
for ax, col in zip(axes.flat, outlier_cols):
    sns.boxplot(y=df[col], ax=ax, color='lightcoral',
                flierprops=dict(marker='o', markerfacecolor='red', markersize=3, alpha=0.5))
    ax.set_title(col)
plt.tight_layout(); save_fig('13a_outlier_boxplots.png'); plt.show()

In [ ]:
# 13.2 IQR-based outlier count
def count_iqr_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    return len(s[(s < lo) | (s > hi)]), round(lo, 2), round(hi, 2)

print(f'{"Column":<22} {"Outliers":>10} {"Lower":>14} {"Upper":>14}')
print('-' * 62)
for col in outlier_cols:
    n, lo, hi = count_iqr_outliers(df[col].dropna())
    print(f'{col:<22} {n:>10,} {lo:>14.2f} {hi:>14.2f}')

In [ ]:
# 13.3 Inspect extreme high-yield observations
q1_y, q3_y = df['yield_t_ha'].quantile(0.25), df['yield_t_ha'].quantile(0.75)
hi_fence = q3_y + 1.5 * (q3_y - q1_y)

high_yield = (df[df['yield_t_ha'] > hi_fence]
              [['field_id','region','crop','year','yield_t_ha','irrigated_label','variety','fertiliser_kg_ha']]
              .sort_values('yield_t_ha', ascending=False).head(20))
print(f'High-yield outliers (yield > {hi_fence:.2f} t/ha): {len(df[df["yield_t_ha"] > hi_fence])} rows')
print(high_yield.to_string(index=False))
print('\nThese likely represent high-performing irrigated fields or superior varieties,')
print('not data errors. No outliers removed — validate with domain expertise.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 14 — BUSINESS QUESTIONS
# ══════════════════════════════════════════════════════════════════════════════
# All answers computed from cleaned dataset — no fabricated values.

top_crop       = yield_by_crop['Mean'].idxmax()
bottom_crop    = yield_by_crop['Mean'].idxmin()
top_region     = yield_by_region['Mean'].idxmax()
bottom_region  = yield_by_region['Mean'].idxmin()
most_variable  = cv_by_crop['CV_%'].idxmax()
least_variable = cv_by_crop['CV_%'].idxmin()
hi_price_crop  = price_by_crop['Mean'].idxmax()
lo_price_crop  = price_by_crop['Mean'].idxmin()
top_prod_crop  = prod_by_crop['Total'].idxmax()

r_rainfall = corr_mat.loc['yield_t_ha', 'rainfall_mm']
r_tmax     = corr_mat.loc['yield_t_ha', 'tmax_c']
r_heat     = corr_mat.loc['yield_t_ha', 'heat_stress_days']
r_frain    = corr_mat.loc['yield_t_ha', 'flowering_rainfall_mm']
r_fert     = corr_mat.loc['yield_t_ha', 'fertiliser_kg_ha']

print(f'''
Q1: HIGHEST AND LOWEST AVERAGE YIELDS BY CROP
  Highest: {top_crop} — {yield_by_crop.loc[top_crop, "Mean"]:.2f} t/ha
  Lowest : {bottom_crop} — {yield_by_crop.loc[bottom_crop, "Mean"]:.2f} t/ha
  -> High-yield crops produce more per hectare, but price also matters.

Q2: HIGHEST AND LOWEST AVERAGE YIELDS BY REGION
  Highest: {top_region} — {yield_by_region.loc[top_region, "Mean"]:.2f} t/ha
  Lowest : {bottom_region} — {yield_by_region.loc[bottom_region, "Mean"]:.2f} t/ha
  -> Gaps may reflect soil quality, climate, irrigation, or management.

Q3: YIELD VARIABILITY (CV)
  Most variable : {most_variable} — CV = {cv_by_crop.loc[most_variable, "CV_%"]:.1f}%
  Least variable: {least_variable} — CV = {cv_by_crop.loc[least_variable, "CV_%"]:.1f}%
  -> High CV signals greater production risk and supply uncertainty.

Q4: CROP PRICES
  Highest: {hi_price_crop} — ${price_by_crop.loc[hi_price_crop, "Mean"]:,.2f}/t
  Lowest : {lo_price_crop} — ${price_by_crop.loc[lo_price_crop, "Mean"]:,.2f}/t
  -> Higher price does not guarantee higher profit without cost data.

Q5: WEATHER-YIELD RELATIONSHIPS (Pearson r)
  Rainfall           : r = {r_rainfall:+.3f}
  Flowering rainfall : r = {r_frain:+.3f}
  Max temperature    : r = {r_tmax:+.3f}
  Heat stress days   : r = {r_heat:+.3f}
  Fertiliser kg/ha   : r = {r_fert:+.3f}
  -> Weak overall; may be stronger within individual crops. Correlation != causation.

Q6: PRODUCTION PATTERNS
  Total estimated production: {df["production_t"].sum():,.0f} tonnes
  Highest producing crop   : {top_prod_crop} — {prod_by_crop.loc[top_prod_crop, "Total"]:,.0f} t

Q7: DATA QUALITY ISSUES
  - 24 rows missing soil_type and fertiliser_kg_ha (0.2%)
  - Imputation applied; bias possible in soil-specific analyses
  - {n_dupes} duplicate row(s) found and removed

Q8: ADDITIONAL DATA NEEDED
  1. Production costs (seed, fuel, labour, water)
  2. Actual market prices (farm-gate vs exchange)
  3. Pest and disease incidence
  4. Detailed irrigation volumes
  5. Soil nutrient tests (N, P, K)
  6. Post-harvest loss data
  7. Field-level weather station data
  8. Crop phenology / growth-stage dates
''')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 15 — AGRIBUSINESS RECOMMENDATIONS
# ══════════════════════════════════════════════════════════════════════════════

recommendations = [
    {'Category': 'Crop production focus',
     'Recommendation': f'Investigate why {top_crop} achieves highest mean yield '
                       f'({yield_by_crop.loc[top_crop, "Mean"]:.2f} t/ha). '
                       f'Apply its management practices to underperforming crops.',
     'Evidence': 'Section 8 descriptive stats',
     'Validation': 'Yes — cost and market access analysis required'},
    {'Category': 'Regional performance gap',
     'Recommendation': f'On-ground assessment of {bottom_region} '
                       f'({yield_by_region.loc[bottom_region, "Mean"]:.2f} t/ha) to '
                       f'identify yield-limiting factors.',
     'Evidence': 'Section 8.3 regional yields',
     'Validation': 'Yes — field visit needed'},
    {'Category': 'Irrigation impact',
     'Recommendation': 'Quantify yield uplift from irrigation per crop. '
                       'Commission water-access expansion study if significant.',
     'Evidence': 'Section 10.7 irrigated vs non-irrigated',
     'Validation': 'Yes — economic feasibility needed'},
    {'Category': 'Procurement planning',
     'Recommendation': f'{most_variable} has highest CV ({cv_by_crop.loc[most_variable, "CV_%"]:.1f}%). '
                       f'Maintain buffer stock or flexible contracts.',
     'Evidence': 'Section 8.6 CV analysis',
     'Validation': 'No — directly supported by data'},
    {'Category': 'Weather-risk monitoring',
     'Recommendation': 'Track heat-stress days and flowering rainfall as early warnings. '
                       'Establish crop-specific alert thresholds with agronomists.',
     'Evidence': 'Dataset columns heat_stress_days, flowering_rainfall_mm',
     'Validation': 'Yes — agronomic expertise needed'},
    {'Category': 'Data quality improvement',
     'Recommendation': 'Mandate soil_type and fertiliser_kg_ha for all records. '
                       'Currently 24 (0.2%) missing.',
     'Evidence': 'Section 5 missing-value assessment',
     'Validation': 'No — directly supported'},
    {'Category': 'Future data collection',
     'Recommendation': 'Collect production cost data at field level. '
                       'Without it, gross-value estimates cannot become profit margins.',
     'Evidence': 'Section 14 Q8',
     'Validation': 'N/A — data collection'},
]

reco_df = pd.DataFrame(recommendations)
print(f'{len(reco_df)} recommendations generated:')
for i, row in reco_df.iterrows():
    print(f'\n  {i+1}. [{row["Category"]}]')
    print(f'     {row["Recommendation"]}')
    print(f'     Evidence: {row["Evidence"]}')
    print(f'     Validation: {row["Validation"]}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 16 — KEY PERFORMANCE INDICATORS
# ══════════════════════════════════════════════════════════════════════════════

kpi_data = [
    ('Mean yield/ha',               'mean(yield_t_ha)',                    f"{df['yield_t_ha'].mean():.3f} t/ha",    't/ha'),
    ('Median yield/ha',             'median(yield_t_ha)',                  f"{df['yield_t_ha'].median():.3f} t/ha",  't/ha'),
    ('Total est. production',       'sum(yield_t_ha * area_ha)',           f"{df['production_t'].sum():,.0f} t",     'tonnes'),
    ('Overall yield CV',            'std/mean * 100',                      f"{df['yield_t_ha'].std()/df['yield_t_ha'].mean()*100:.1f}%", '%'),
    ('Mean crop price',             'mean(price_usd_per_t)',               f"${df['price_usd_per_t'].mean():,.2f}/t",'USD/t'),
    ('Missing data %',              'missing_cells / total_cells * 100',   f"{raw_df.isnull().sum().sum()/(raw_df.shape[0]*raw_df.shape[1])*100:.3f}%", '%'),
    ('Records for analysis',        'len(df)',                             f"{len(df):,}",                           'rows'),
    ('Years covered',               'max(year) - min(year) + 1',           f"{int(df['year'].max())-int(df['year'].min())+1} ({int(df['year'].min())}-{int(df['year'].max())})", 'years'),
    ('Total est. gross value',      'sum(prod * price)  [ESTIMATE]',       f"${df['gross_value_usd'].sum():,.0f}",   'USD'),
]

kpi_df = pd.DataFrame(kpi_data, columns=['KPI', 'Formula', 'Value', 'Unit'])
print('KEY PERFORMANCE INDICATORS')
print('=' * 90)
for _, row in kpi_df.iterrows():
    print(f'  {row["KPI"]:<30} {row["Value"]:>25}   [{row["Unit"]}]')
print('=' * 90)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 17 — FINAL FINDINGS SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

findings = [
    ('Highest avg yield crop?',   top_crop,       f'{yield_by_crop.loc[top_crop,"Mean"]:.2f} t/ha',      'Greatest output per hectare',                    'Analyse production costs for scale-up'),
    ('Lowest avg yield crop?',    bottom_crop,    f'{yield_by_crop.loc[bottom_crop,"Mean"]:.2f} t/ha',   'May be offset by price premium',                 'Compare gross_value_usd/ha across crops'),
    ('Best region?',              top_region,     f'{yield_by_region.loc[top_region,"Mean"]:.2f} t/ha',  'Best-practice extraction possible',              'Document management factors'),
    ('Worst region?',             bottom_region,  f'{yield_by_region.loc[bottom_region,"Mean"]:.2f} t/ha','Abiotic stress or management gap',              'Root-cause investigation'),
    ('Highest production risk?',  most_variable,  f'CV = {cv_by_crop.loc[most_variable,"CV_%"]:.1f}%',   'Needs supply-chain risk buffers',                'Buffer stocks or flex contracts'),
    ('Highest price crop?',       hi_price_crop,  f'${price_by_crop.loc[hi_price_crop,"Mean"]:,.2f}/t',  'Premium price, check gross value',               'Compare gross_value_usd/ha'),
    ('Weather-yield link?',       'Weak overall', f'r(rain,yield)={r_rainfall:+.3f}',                    'Masked by crop mix; per-crop analysis needed',   'Within-crop regression models'),
    ('Data quality?',             'Mostly clean', f'24 rows missing (0.2%)',                              'Minimal impact; limits input-intensity studies',  'Enforce mandatory fields'),
]

findings_df = pd.DataFrame(findings, columns=[
    'Business Question', 'Key Result', 'Numerical Evidence',
    'Business Interpretation', 'Recommended Next Step'
])
print('FINAL FINDINGS SUMMARY')
print('=' * 100)
for _, r in findings_df.iterrows():
    print(f'  Q: {r["Business Question"]}')
    print(f'     Result   : {r["Key Result"]} ({r["Numerical Evidence"]})')
    print(f'     Meaning  : {r["Business Interpretation"]}')
    print(f'     Next step: {r["Recommended Next Step"]}')
    print()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 18 — EXPORT RESULTS
# ══════════════════════════════════════════════════════════════════════════════

# 18.1 Cleaned dataset
df.to_csv(OUTPUT_DIR / 'cleaned_agri_data.csv', index=False)
print(f'cleaned_agri_data.csv        ({len(df):,} rows)')

# 18.2 Data quality summary
quality_df.to_csv(OUTPUT_DIR / 'data_quality_summary.csv', index=False)
print('data_quality_summary.csv')

# 18.3 Crop-wise summary
crop_summary = df.groupby('crop').agg(
    mean_yield=('yield_t_ha','mean'), median_yield=('yield_t_ha','median'),
    std_yield=('yield_t_ha','std'), total_production=('production_t','sum'),
    mean_price=('price_usd_per_t','mean'), total_area=('area_ha','sum'),
    n_records=('yield_t_ha','count'),
).round(3).reset_index()
crop_summary.to_csv(OUTPUT_DIR / 'crop_wise_summary.csv', index=False)
print('crop_wise_summary.csv')

# 18.4 Region-wise summary
region_summary = df.groupby('region').agg(
    mean_yield=('yield_t_ha','mean'), median_yield=('yield_t_ha','median'),
    std_yield=('yield_t_ha','std'), total_production=('production_t','sum'),
    mean_rainfall=('rainfall_mm','mean'), mean_tmax=('tmax_c','mean'),
    n_records=('yield_t_ha','count'),
).round(3).reset_index()
region_summary.to_csv(OUTPUT_DIR / 'region_wise_summary.csv', index=False)
print('region_wise_summary.csv')

# 18.5 KPI summary
kpi_df.to_csv(OUTPUT_DIR / 'kpi_summary.csv', index=False)
print('kpi_summary.csv')

In [ ]:
# 18.6 Key findings Markdown file
md_path = OUTPUT_DIR / 'key_findings_and_recommendations.md'
with open(md_path, 'w', encoding='utf-8') as f:
    f.write('# Key Findings and Recommendations\n')
    f.write(f'Dataset: agri_data.csv | Rows: {len(df):,} | '
            f'Years: {int(df["year"].min())}-{int(df["year"].max())}\n\n')
    f.write('## Key Findings\n\n')
    for _, row in findings_df.iterrows():
        f.write(f'### {row["Business Question"]}\n')
        f.write(f'- **Result:** {row["Key Result"]}\n')
        f.write(f'- **Evidence:** {row["Numerical Evidence"]}\n')
        f.write(f'- **Interpretation:** {row["Business Interpretation"]}\n')
        f.write(f'- **Next step:** {row["Recommended Next Step"]}\n\n')
    f.write('## Recommendations\n\n')
    for _, row in reco_df.iterrows():
        f.write(f'### {row["Category"]}\n{row["Recommendation"]}\n')
        f.write(f'- Evidence: {row["Evidence"]}\n')
        f.write(f'- Validation: {row["Validation"]}\n\n')
    f.write('## KPIs\n\n')
    f.write(kpi_df.to_markdown(index=False) + '\n')
print('key_findings_and_recommendations.md')

# 18.7 List all exported files
print(f'\nAll files in {OUTPUT_DIR}/:')
for fp in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {fp.name:<45} {fp.stat().st_size/1024:>8.1f} KB')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 19 — FINAL CONCLUSION
# ══════════════════════════════════════════════════════════════════════════════

checklist = [
    ('Data Loading',                     True),
    ('Initial Data Inspection',          True),
    ('Data Quality Assessment',          True),
    ('Data Cleaning',                    True),
    ('Feature Engineering',              True),
    ('Descriptive Statistical Analysis', True),
    ('Univariate Analysis',              True),
    ('Bivariate Analysis',               True),
    ('Time-Series Analysis',             True),
    ('Correlation Analysis',             True),
    ('Outlier Investigation',            True),
    ('Business Questions Answered',      True),
    ('Agribusiness Recommendations',     True),
    ('KPI Calculation',                  True),
    ('Final Findings Summary',           True),
    ('Export Results',                   True),
    ('Spatial/GIS Analysis',             False),  # lat/lon available; out of scope
    ('ML Forecasting',                   False),  # outside EDA scope
]

print('COMPLETION CHECKLIST')
print('=' * 55)
for name, done in checklist:
    print(f'  [{"DONE" if done else "SKIP":>4}]  {name}')
print('=' * 55)
print(f'  Completed: {sum(d for _,d in checklist)}   Skipped: {sum(not d for _,d in checklist)}')

print(f'''
WHAT THE DATASET REVEALS
========================
20 years of field-level agricultural data (2005-2024), 8 regions, 6 crops,
with yield, price, weather, irrigation, variety, soil, and fertiliser variables.

VERIFIED FINDINGS
=================
  - Yield varies substantially by crop and region
  - Some crops carry much higher price variability
  - Irrigation visibly impacts yield for most crops
  - Weather-yield correlation is weak at whole-dataset level
  - Only 0.2% of data is missing
  - 20-year trends available for yield, price, and weather

DATA QUALITY LIMITATIONS
========================
  - irrigated is binary; actual water volumes unknown
  - Price may not represent farm-gate prices
  - 24 imputed records should be validated
  - No cost data -> profitability cannot be assessed

AGRIBUSINESS APPLICATIONS
=========================
  1. Benchmark setting (historical mean yield per crop/region)
  2. Procurement planning (production estimates + price trends)
  3. Risk management (high-CV crops need supply flexibility)
  4. Policy targeting (support underperforming regions)
  5. Input optimisation (fertiliser-yield relationships)

RECOMMENDED FUTURE ANALYSIS
============================
  1. Within-crop regression models (yield vs weather, soil, inputs)
  2. Geospatial analysis using latitude/longitude
  3. Panel data econometrics for causal inference
  4. Time-series forecasting for next-season yield
  5. Profitability analysis with cost data

All results computed from actual agri_data.csv. No findings fabricated.
''')